# OCSD, one scene traced module by module (for the slides)

Generates **one** image with the final OCSD (M1–M5 on SDXL, setting "natural", the exact code and settings of the
72-scene paper run) and saves what every module produces: M1 parsed objects, M2 candidates and the selected object,
M4 composed foreground and blending masks, M5 attention maps, energy, the latent at every denoising step and the
predicted clean image, M5(d) detection check and repair. Nothing in the computation is changed; the code only records.

**How to run:** Runtime → Change runtime type → **L4 GPU** (or A100) → **Run all**. About 15–20 minutes (most of it is the first
model download). At the end a ZIP downloads; it is also on Drive at `KLTN_OCSD/trace_ocsd/`.

In [ ]:
#@title 1. Settings
SID    = "qd_3_medium_003"   #@param ["qd_3_medium_003", "qd_1_medium_003", "qd_5_complex_003", "qd_8p_complex_003"]
SEED   = 0                   #@param {type:"integer"}
BRANCH = "claude/project-thread-srtlwl"  #@param {type:"string"}
DRIVE_ROOT = "/content/drive/MyDrive/KLTN_OCSD"  #@param {type:"string"}
# qd_3_medium_003 = "an orange bus, a wooden bench and an orange bus near a train station" (the scene on the results slide)

In [ ]:
#@title 2. Drive, GPU check, code, libraries
import os, subprocess, sys, shutil
from google.colab import drive
drive.mount("/content/drive")
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"], capture_output=True, text=True)
print(gpu.stdout)
if gpu.returncode or int(gpu.stdout.strip().split(",")[-1]) < 20000:
    raise RuntimeError("This needs a GPU with at least 20 GB (L4 or A100): Runtime > Change runtime type > L4 GPU, then Run all")
# one OCSD image peaked at 18.8 GB in the paper run, so an L4 (24 GB) is enough; it is just slower than an A100
CODE = "/content/KLTN_OCSD_trace"
shutil.rmtree(CODE, ignore_errors=True)
subprocess.run(["git", "clone", "-q", "-b", BRANCH, "https://github.com/bobbibao/ocsd-thesis.git", CODE], check=True)
print("Code:", subprocess.run(["git", "-C", CODE, "log", "-1", "--format=%H %s"], capture_output=True, text=True).stdout)
sys.path.insert(0, CODE)
for _m in [m for m in sys.modules if m == "ocsd" or m.startswith("ocsd.")]: del sys.modules[_m]
os.environ["HF_HOME"] = "/content/hf_cache"
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=True)
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", os.path.join(CODE, "requirements-colab.txt")], capture_output=True, text=True)
print(r.stdout[-1000:], r.stderr[-2000:])
if r.returncode:
    raise RuntimeError("pip install failed (see the text above)")
from ocsd import hfcache
hfcache.repair(os.environ["HF_HOME"])

In [ ]:
#@title 3. Generate the image and record every module (about 3 minutes after the models are downloaded)
from ocsd import trace_xl
META = trace_xl.trace(DRIVE_ROOT, sid=SID, seed=SEED)
print(META.get("paper_run_check"))

In [ ]:
#@title 4. Show the result and download the ZIP
from IPython.display import Image, display
out = os.path.join(DRIVE_ROOT, "trace_ocsd", f"{SID}_s{SEED}")
display(Image(os.path.join(out, "NOISE_TO_IMAGE.jpg"), width=1400))
display(Image(os.path.join(out, "OVERVIEW.jpg"), width=1400))
from google.colab import files
try:
    files.download(META["zip"])
except Exception as exc:
    print("Get it from Drive:", META["zip"], exc)
print("Done. Disconnect the GPU runtime.")